---
title: "Extracting structured data from papers using LLM"
author: 
  - name: Marian Klose
    orcid: 0009-0005-1706-6289
    email: marian.klose@fu-berlin.de
    affiliations:
      - name: Freie Universität Berlin
date: now
callout-appearance: minimal
execute:
  enabled: true
  echo: true
  message: true
  warning: true
format:
  html:
    toc: true
    number-sections: true
    embed-resources: true
---


This notebook shows a proof of concept how to use a large language model (LLM) to extract structured data about the underlying model from NLME publications. 

## Preamble

In [1]:
# load packages
import pandas as pd
import json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

## General settings

We go for the cheapest model `gpt-5-nano` to keep the costs low. The costs per 1M tokens are defined so that the total costs can be calculated later on.

In [2]:
# define gpt model
gpt_model = "gpt-5-nano"

# define input and output costs (https://platform.openai.com/docs/pricing?latest-pricing=standard)
input_cost_pertoken_usd = 0.005/1_000_000  # cost per token for input
output_cost_pertoken_usd = 0.40/1_000_000  # cost per token for output

# define number of papers to process
n_papers = 200

## Read in data

We read back in the previously saved (enriched) PMC query results including abstract and full body text.

In [3]:
# read in json file with enriched article records
articles = pd.read_json("out/pmc_query_results_enriched.json", orient="records")

# show articles with prettyr print
articles.head()

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,Abstract,Body
0,40687901,Pharmacokinetics and Safety with Bioequivalenc...,"Wu B, Wang W, Zhang Q, Yu G, Lin J, Zhang T, Y...",Drug Des Devel Ther. 2025 Jul 14;19:6025-6035....,Wu B,Drug Des Devel Ther,2025,2025/07/21,PMC12274271,None,10.2147/DDDT.S508202,PURPOSE: Isosorbide mononitrate was recommende...,pmc Drug Des Devel Ther Drug Des Devel Ther 95...
1,32034814,Evaluation of pharmacokinetics and safety with...,"Wang T, Wang Y, Lin S, Fang L, Lou S, Zhao D, ...",J Clin Lab Anal. 2020 Jun;34(6):e23228. doi: 1...,Wang T,J Clin Lab Anal,2020,2020/02/09,PMC7307347,None,10.1002/jcla.23228,"BACKGROUND AND OBJECTIVE: Amlodipine, a main s...",J Clin Lab Anal J. Clin. Lab. Anal 3636 jclinl...
2,30710324,Systemic Bioequivalence Is Unlikely to Equal T...,"Au JL, Lu Z, Abbiati RA, Wientjes MG.",AAPS J. 2019 Feb 1;21(2):24. doi: 10.1208/s122...,Au JL,AAPS J,2019,2019/02/03,PMC6432930,NIHMS1009981,10.1208/s12248-019-0296-z,Approval of generic drugs by the US Food and D...,AAPS J AAPS J 319 nihpa The AAPS journal 1550-...
3,40781573,Assessment of Pharmacokinetics and Safety with...,"Fu Q, Huang C, Yuan Y, Wang Y, Zhu B, Liu Y, T...",Drugs R D. 2025 Sep;25(3):263-274. doi: 10.100...,Fu Q,Drugs R D,2025,2025/08/09,PMC12460201,None,10.1007/s40268-025-00519-4,"BACKGROUND AND OBJECTIVE: Nitroglycerin, a cor...",pmc Drugs R D Drugs R D 2559 drugsrd Drugs in ...
4,24151591,Pharmacokinetics and bioequivalence evaluation...,"Brioschi TM, Schramm SG, Kano EK, Koono EE, Ch...",Biomed Res Int. 2013;2013:281392. doi: 10.1155...,Brioschi TM,Biomed Res Int,2013,2013/10/24,PMC3787571,None,10.1155/2013/281392,The purpose of this study was to investigate c...,Biomed Res Int Biomed Res Int 2029 bmri BMRI B...


## Initialize OpenAI client

A client for the OpenAI API is initialized using an API key stored as an environment variable.

In [4]:
# initialize openai client
client = OpenAI()

## Assess suitability of publications

The first task is to assess whether a publication is suitable for data extraction. Publications are suitable if they report a single population pharmacokinetic model. Other types of publications (reviews, tutorials, or studies reporting multiple models) are probably too complex to be handled by the LLM in a reliable way.

### Define structured output classes

Structured output classes are defined using `pydantic`. The classes define the expected structure of the output data and help to validate the extracted information. More information can be found under https://platform.openai.com/docs/guides/structured-outputs. 

In [5]:
# single class for suitability flag
class SuitableFlag(BaseModel):
    suitable: bool = Field(
        description='''
        Indicates if this paper reports a single bioequivalence trial (True). 
        If the article represents a comment, a perspective, a review, a correction, a retraction,
        or a meta-analysis, it is not suitable (False). It is also not suitable if 
        it tackles some methodological issue without reporting a specific trial. It has to report
        the results of a single bioequivalence trial.
        '''
    )

### Define system message

The system message defines the role of the LLM and provides general instructions on how to handle the task.

In [6]:
# define system message
system_message_flag = '''
You are a pharmacometric / clinical pharmacology expert. You will be provided with a title and abstract of a scientific publication. 
Your task is to classify whether the paper reports a single bioequivalence trial (True) or not. 
If the article represents a comment, a perspective, a review, a correction, a retraction,
or a meta-analysis, it is not suitable (False). It is also not suitable if 
it tackles some methodological issue without reporting a specific trial. It has to report
the results of a single bioequivalence trial.
''' 

# print
system_message_flag

'\nYou are a pharmacometric / clinical pharmacology expert. You will be provided with a title and abstract of a scientific publication. \nYour task is to classify whether the paper reports a single bioequivalence trial (True) or not. \nIf the article represents a comment, a perspective, a review, a correction, a retraction,\nor a meta-analysis, it is not suitable (False). It is also not suitable if \nit tackles some methodological issue without reporting a specific trial. It has to report\nthe results of a single bioequivalence trial.\n'

### Make requests to LLM

We now loop over all publications and make requests to the LLM to assess the suitability of each publication. The title and abstract are provided as context to the LLM. The responses are parsed and saved in a new column in the dataframe (suitable or not suitable).

In [7]:
# Initialize new columns
articles["Suitable"] = None
articles["InputTokensFlag"] = None
articles["OutputTokensFlag"] = None
articles["PriceUSDFlag"] = None

# Empty lists to track totals
input_tokens_flag = []
output_tokens_flag = []
prices_usd_flag = []

# loop over elements in list
for idx, row in articles.head(n_papers).iterrows():
    # message to user
    print(f"Processing article {idx+1}/{len(articles)}: {row['Title']}")
    
    # extract information
    title = row["Title"],
    abstract = row["Abstract"]
    
    # construct user message
    user_message = f'title: {title}\nabstract: {abstract}'
    
    # try to make request to chatgpt
    try:
        # make request
        response = client.responses.parse(
            model=gpt_model,
            input=[
                {"role": "system", "content": system_message_flag},
                {"role": "user", "content": user_message},
            ],
            text_format=SuitableFlag
        )
        
        # retrieve and print the response
        model_info = response.output_parsed
        print(f"> Suitable? {model_info.suitable}\n")
        
        # append tokens and prices to lists
        in_toks = response.usage.input_tokens
        out_toks = response.usage.output_tokens
        price = (in_toks * input_cost_pertoken_usd) + (out_toks * output_cost_pertoken_usd)
        
        # update lists
        input_tokens_flag.append(in_toks)
        output_tokens_flag.append(out_toks)
        prices_usd_flag.append(price)
        
        # update dataframe
        articles.at[idx, "Suitable"] = model_info.suitable
        articles.at[idx, "InputTokensFlag"] = in_toks
        articles.at[idx, "OutputTokensFlag"] = out_toks
        articles.at[idx, "PriceUSDFlag"] = price
        
    # catch exceptions
    except Exception as e:
        print(f"Error: {e}")
        
# define prices for these calls (in USD)
total_price_flag = sum(prices_usd_flag)
print(f"Total price for flagging {n_papers} papers: ${total_price_flag:.3f}")

Processing article 1/200: Pharmacokinetics and Safety with Bioequivalence of Isosorbide Mononitrate Sustained-Release Tablets in Chinese Healthy Volunteers: Bioequivalence Study


> Suitable? True

Processing article 2/200: Evaluation of pharmacokinetics and safety with bioequivalence of Amlodipine in healthy Chinese volunteers: Bioequivalence Study Findings


> Suitable? False

Processing article 3/200: Systemic Bioequivalence Is Unlikely to Equal Target Site Bioequivalence for Nanotechnology Oncologic Products


> Suitable? False

Processing article 4/200: Assessment of Pharmacokinetics and Safety with Bioequivalence of the Nitroglycerin Sublingual Tablets of Two Formulations in Chinese Healthy Subjects: A Bioequivalence Study


> Suitable? True

Processing article 5/200: Pharmacokinetics and bioequivalence evaluation of cyclobenzaprine tablets


...

> Suitable? True

Processing article 197/200: Bioequivalence study of three ibuprofen formulations after single dose administration in healthy volunteers


> Suitable? True

Processing article 198/200: Comparative Pharmacokinetics, Bioequivalence and Safety Study of Two Recombinant Human Chorionic Gonadotropin Injections in Healthy Chinese Subjects


> Suitable? True

Processing article 199/200: Bioequivalence of oral products and the biopharmaceutics classification system: science, regulation, and public policy


> Suitable? False

Processing article 200/200: Bioequivalence study of fluticasone propionate nebuliser suspensions in healthy Chinese subjects


> Suitable? True

Total price for flagging 200 papers: $0.052


### Show updated dataframe

In [8]:
# show updated dataframe
articles.head(n_papers)

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,Abstract,Body,Suitable,InputTokensFlag,OutputTokensFlag,PriceUSDFlag
0,40687901,Pharmacokinetics and Safety with Bioequivalenc...,"Wu B, Wang W, Zhang Q, Yu G, Lin J, Zhang T, Y...",Drug Des Devel Ther. 2025 Jul 14;19:6025-6035....,Wu B,Drug Des Devel Ther,2025,2025/07/21,PMC12274271,None,10.2147/DDDT.S508202,PURPOSE: Isosorbide mononitrate was recommende...,pmc Drug Des Devel Ther Drug Des Devel Ther 95...,True,766,1360,0.000548
1,32034814,Evaluation of pharmacokinetics and safety with...,"Wang T, Wang Y, Lin S, Fang L, Lou S, Zhao D, ...",J Clin Lab Anal. 2020 Jun;34(6):e23228. doi: 1...,Wang T,J Clin Lab Anal,2020,2020/02/09,PMC7307347,None,10.1002/jcla.23228,"BACKGROUND AND OBJECTIVE: Amlodipine, a main s...",J Clin Lab Anal J. Clin. Lab. Anal 3636 jclinl...,False,831,656,0.000267
2,30710324,Systemic Bioequivalence Is Unlikely to Equal T...,"Au JL, Lu Z, Abbiati RA, Wientjes MG.",AAPS J. 2019 Feb 1;21(2):24. doi: 10.1208/s122...,Au JL,AAPS J,2019,2019/02/03,PMC6432930,NIHMS1009981,10.1208/s12248-019-0296-z,Approval of generic drugs by the US Food and D...,AAPS J AAPS J 319 nihpa The AAPS journal 1550-...,False,597,272,0.000112
3,40781573,Assessment of Pharmacokinetics and Safety with...,"Fu Q, Huang C, Yuan Y, Wang Y, Zhu B, Liu Y, T...",Drugs R D. 2025 Sep;25(3):263-274. doi: 10.100...,Fu Q,Drugs R D,2025,2025/08/09,PMC12460201,None,10.1007/s40268-025-00519-4,"BACKGROUND AND OBJECTIVE: Nitroglycerin, a cor...",pmc Drugs R D Drugs R D 2559 drugsrd Drugs in ...,True,881,592,0.000241
4,24151591,Pharmacokinetics and bioequivalence evaluation...,"Brioschi TM, Schramm SG, Kano EK, Koono EE, Ch...",Biomed Res Int. 2013;2013:281392. doi: 10.1155...,Brioschi TM,Biomed Res Int,2013,2013/10/24,PMC3787571,None,10.1155/2013/281392,The purpose of this study was to investigate c...,Biomed Res Int Biomed Res Int 2029 bmri BMRI B...,True,607,336,0.000137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,32338459,Bioequivalence and pharmacodynamics of a gener...,"Li X, Liu L, Xu B, Xiang Q, Li Y, Zhang P, Wan...",Pharmacol Res Perspect. 2020 Apr;8(2):e00593. ...,Li X,Pharmacol Res Perspect,2020,2020/04/28,PMC7184321,None,10.1002/prp2.593,To assess bioequivalence of a generic dabigatr...,Pharmacol Res Perspect Pharmacol Res Perspect ...,True,762,720,0.000292
196,18959779,Bioequivalence study of three ibuprofen formul...,"Bramlage P, Goldis A.",BMC Pharmacol. 2008 Oct 29;8:18. doi: 10.1186/...,Bramlage P,BMC Pharmacol,2008,2008/10/31,PMC2613135,None,10.1186/1471-2210-8-18,BACKGROUND: This phase I study was designed to...,BMC Pharmacol 56 bmcphar BMC Pharmacology 1471...,True,687,400,0.000163
197,32099328,"Comparative Pharmacokinetics, Bioequivalence a...","Wang J, Yang T, Mei H, Yu X, Peng H, Wang R, C...",Drug Des Devel Ther. 2020 Jan 29;14:435-444. d...,Wang J,Drug Des Devel Ther,2020,2020/02/27,PMC6996485,None,10.2147/DDDT.S235064,OBJECTIVE: To evaluate the pharmacokinetics (P...,Drug Des Devel Ther Drug Des Devel Ther 958 dd...,True,751,400,0.000164
198,21775984,Bioequivalence of oral products and the biopha...,"Amidon KS, Langguth P, Lennernäs H, Yu L, Amid...",Clin Pharmacol Ther. 2011 Sep;90(3):467-70. do...,Amidon KS,Clin Pharmacol Ther,2011,2011/07/22,PMC3228645,NIHMS337891,10.1038/clpt.2011.109,The demonstration of bioequivalence (BE) is an...,Clin Pharmacol Ther 319 nihpa Clinical pharmac...,False,393,656,0.000264


## Retrieve NLME model information

Now we can focus on the publications that were classified as suitable and try to extract structured information about the underlying NLME model.

### Filter dataframe

First, we filter the dataframe to only include publications that were classified as suitable.

In [9]:
# filter dataframe for suitable articles
articles_suitable = articles[articles["Suitable"] == True].reset_index(drop=True)

# show filtered dataframe
articles_suitable.head()

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,Abstract,Body,Suitable,InputTokensFlag,OutputTokensFlag,PriceUSDFlag
0,40687901,Pharmacokinetics and Safety with Bioequivalenc...,"Wu B, Wang W, Zhang Q, Yu G, Lin J, Zhang T, Y...",Drug Des Devel Ther. 2025 Jul 14;19:6025-6035....,Wu B,Drug Des Devel Ther,2025,2025/07/21,PMC12274271,None,10.2147/DDDT.S508202,PURPOSE: Isosorbide mononitrate was recommende...,pmc Drug Des Devel Ther Drug Des Devel Ther 95...,True,766,1360,0.000548
1,40781573,Assessment of Pharmacokinetics and Safety with...,"Fu Q, Huang C, Yuan Y, Wang Y, Zhu B, Liu Y, T...",Drugs R D. 2025 Sep;25(3):263-274. doi: 10.100...,Fu Q,Drugs R D,2025,2025/08/09,PMC12460201,None,10.1007/s40268-025-00519-4,"BACKGROUND AND OBJECTIVE: Nitroglycerin, a cor...",pmc Drugs R D Drugs R D 2559 drugsrd Drugs in ...,True,881,592,0.000241
2,24151591,Pharmacokinetics and bioequivalence evaluation...,"Brioschi TM, Schramm SG, Kano EK, Koono EE, Ch...",Biomed Res Int. 2013;2013:281392. doi: 10.1155...,Brioschi TM,Biomed Res Int,2013,2013/10/24,PMC3787571,None,10.1155/2013/281392,The purpose of this study was to investigate c...,Biomed Res Int Biomed Res Int 2029 bmri BMRI B...,True,607,336,0.000137
3,28706331,Bioequivalence of generic and branded amoxicil...,"Pathak P, Pandit VA, Dhande PP.",Indian J Pharmacol. 2017 Mar-Apr;49(2):176-181...,Pathak P,Indian J Pharmacol,2017,2017/07/15,PMC5497440,None,10.4103/ijp.IJP_793_16,CONTEXT: The Medical Council of India urges do...,Indian J Pharmacol Indian J Pharmacol 979 ijph...,True,683,336,0.000138
4,22087062,Bioequivalence assessment of two formulations ...,"Al-Talla ZA, Akrawi SH, Tolley LT, Sioud SH, Z...",Drug Des Devel Ther. 2011;5:427-33. doi: 10.21...,Al-Talla ZA,Drug Des Devel Ther,2011,2011/11/17,PMC3210071,None,10.2147/DDDT.S24504,BACKGROUND: This study assessed the relative b...,"Drug Des Devel Ther 958 dddt Drug Design, Deve...",True,554,656,0.000265


### Define structured output classes

We now define a set of structured output classes to capture the relevant information about the NLME model. The classes are again defined using `pydantic` and include various fields for the fields of interest.

In [10]:
class AdministrationType(str, Enum):
    IV = "intravenous"
    ORAL = "oral"
    IM = "intramuscular"
    SC = "subcutaneous"
    OTHER = "other"
    NOT_REPORTED = "not reported"


class FormulationType(str, Enum):
    TABLET = "tablet"
    CAPSULE = "capsule"
    SOLUTION = "solution"
    SUSPENSION = "suspension"
    POWDER = "powder"
    OTHER = "other"
    NOT_REPORTED = "not reported"

class PKSummary(BaseModel):
    """Central tendency and dispersion for a PK parameter."""
    central_tendency: Optional[float] = Field(
        description="Mean or median of the PK parameter for the TEST product. "
                    "Enter numeric value, not rounded. Enter 'missing' if missing.",
    )
    central_tendency_type: Optional[str] = Field(
        description="Specify whether the central tendency is a 'mean' or 'median'. "
                    "Enter 'missing' if not reported."
    )
    central_tendency_unit: Optional[str] = Field(
        description="Unit of the central tendency (e.g., ng/mL, h, ng·h/mL). "
                    "Enter 'missing' if not provided."
    ) 
    dispersion: Optional[str] = Field(
        description="Numerical value for the dispersion. Range, IQR, SD, or CI associated with the PK parameter. E.g., '12-24', '4-9', '5', '15'."
                    "Enter 'missing' if not reported."
    )
    dispersion_type: Optional[str] = Field(
        description="Specify which type of dispersion it is (e.g., 'range', 'IQR'). "
                    "Enter 'missing' if not reported."
    )   
    dispersion_unit: Optional[str] = Field(
        description="Unit of the dispersion (e.g., ng/mL, h, ng·h/mL, %). "
                    "Enter 'missing' if not provided."
    )


class PKParameters(BaseModel):
    Cmax: PKSummary = Field(description="Cmax results for TEST formulation only. Fasting only if there are both fasting and fed results.")
    Tmax: PKSummary = Field(description="Tmax results for TEST formulation only.  Fasting only if there are both fasting and fed results.")
    AUC0_t: PKSummary = Field(description="AUC0-t results for TEST formulation only.  Fasting only if there are both fasting and fed results.")
    AUC0_inf: PKSummary = Field(description="AUC0-inf results for TEST formulation only.  Fasting only if there are both fasting and fed results.")


class BioequivalenceTrial(BaseModel):
    compound: str = Field(
        description="Name of the compound used in the bioequivalence trial."
    )
    
    administration_route: AdministrationType = Field(
        description="Route of administration (oral, IV, IM, etc.). "
                    "Enter 'not reported' if unavailable."
    )
    
    formulation: FormulationType = Field(
        description="Formulation type for the TEST product (tablet, capsule, etc.). "
                    "Enter 'not reported' if unavailable."
    )
    
    dose: Optional[float] = Field(
        description="Administered dose of the TEST product. Numeric only. "
                    "Enter 'missing' if not provided."
    )
    dose_unit: Optional[str] = Field(
        description="Unit of the dose (e.g., mg, mcg). Enter 'missing' if not provided."
    )
    
    number_of_patients: Optional[int] = Field(
        description="Total number of subjects included in the trial. "
                    "Enter 'missing' if not provided."
    )
    
    last_sample_time: Optional[str] = Field(
        description="Time of the last sample taken without unit (e.g., '24'). "
                    "Enter 'missing' if not provided."
    )
    
    last_sample_time_unit: Optional[str] = Field(
        description="Unit of the last sample time (e.g., 'h', 'days'). "
                    "Enter 'missing' if not provided."
    )

    pk_parameters: PKParameters = Field(
        description="Reported PK results (central tendency + dispersion) for TEST product only."
    )

### Define system message and user message

A different system message is defined for the new task.

In [11]:
# define system message
system_message_content = '''
You are a pharmacometric / clinical pharmacology expert at extracting structured information about bioequivalence trials from unstructured text. 
You will be provided with a scientific publication that reports a single bioequivalence trial. 
Your task is to extract key information about the published trial from the results section of that paper, such as
the compound name, administration route, formulation type, dose, number of patients, last sample time,
and PK parameters (Cmax, Tmax, AUC0-t, AUC0-inf) for the investigated test formulation only.
''' 

# print
system_message_content

'\nYou are a pharmacometric / clinical pharmacology expert at extracting structured information about bioequivalence trials from unstructured text. \nYou will be provided with a scientific publication that reports a single bioequivalence trial. \nYour task is to extract key information about the published trial from the results section of that paper, such as\nthe compound name, administration route, formulation type, dose, number of patients, last sample time,\nand PK parameters (Cmax, Tmax, AUC0-t, AUC0-inf) for the investigated test formulation only.\n'

### Make request to LLM

We now loop over the filtered dataframe `articles_suitable` and make requests to the LLM to extract structured information about the NLME model. Now we provide the (more costly) full body text as context to the LLM. The responses are parsed and saved as JSON.

In [12]:
# Initialize new columns
articles["InputTokensContent"] = None
articles["OutputTokensContent"] = None
articles["PriceUSDContent"] = None

# Empty lists to track totals
input_tokens_content = []
output_tokens_content = []
prices_usd_content = []

# list to collect all extracted model reports
model_reports = []

# loop over elements in list
for idx, row in articles_suitable.head(n_papers).iterrows():
    # message to user
    print(f"Processing article {idx+1}/{len(articles_suitable)}: {row['Title']}")
    
    # extract information (user message)
    user_message_content = row["Body"]
    
    # try to make request to chatgpt
    try:
        # make request
        response = client.responses.parse(
            model=gpt_model,
            input=[
                {"role": "system", "content": system_message_content},
                {"role": "user", "content": user_message_content},
            ],
            text_format=BioequivalenceTrial
        )
        
        # retrieve and print the response
        model_content = response.output_parsed
        
        # convert output to dict
        model_dict = model_content.model_dump()
        
        # add metadata from dataframe (identifiers etc.)
        model_dict.update({
            "pmid": row.get("PMID", None),
            "pmcid": row.get("PMCID", None),
            "year": row.get("Publication Year", None),
            "title": row.get("Title", None),
            "abstract": row.get("Abstract", None)
        })
        
        # append to model_reports list
        model_reports.append(model_dict)
        
        # print confirmation
        print(f"> Added to model reports.\n")
        
        # append tokens and prices to lists
        in_toks = response.usage.input_tokens
        out_toks = response.usage.output_tokens
        price = (in_toks * input_cost_pertoken_usd) + (out_toks * output_cost_pertoken_usd)
        
        # update lists
        input_tokens_content.append(in_toks)
        output_tokens_content.append(out_toks)
        prices_usd_content.append(price)
        
    # catch exceptions
    except Exception as e:
        print(f"Error: {e}")
        
# define prices for these calls (in USD)
total_price_content = sum(prices_usd_content)
print(f"Total price for extracting content from {len(articles_suitable)} papers: ${total_price_content:.3f}")

# save to json
output_path = "out/extracted_pk_information.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(model_reports, f, indent=2, ensure_ascii=False)

Processing article 1/114: Pharmacokinetics and Safety with Bioequivalence of Isosorbide Mononitrate Sustained-Release Tablets in Chinese Healthy Volunteers: Bioequivalence Study


> Added to model reports.

Processing article 2/114: Assessment of Pharmacokinetics and Safety with Bioequivalence of the Nitroglycerin Sublingual Tablets of Two Formulations in Chinese Healthy Subjects: A Bioequivalence Study


> Added to model reports.

Processing article 3/114: Pharmacokinetics and bioequivalence evaluation of cyclobenzaprine tablets


> Added to model reports.

Processing article 4/114: Bioequivalence of generic and branded amoxicillin capsules in healthy human volunteers


> Added to model reports.

Processing article 5/114: Bioequivalence assessment of two formulations of ibuprofen


...

> Added to model reports.

Processing article 111/114: Bioequivalence and pharmacodynamics of a generic dabigatran etexilate capsule in healthy Chinese subjects under fasting and fed conditions


> Added to model reports.

Processing article 112/114: Bioequivalence study of three ibuprofen formulations after single dose administration in healthy volunteers


> Added to model reports.

Processing article 113/114: Comparative Pharmacokinetics, Bioequivalence and Safety Study of Two Recombinant Human Chorionic Gonadotropin Injections in Healthy Chinese Subjects


> Added to model reports.

Processing article 114/114: Bioequivalence study of fluticasone propionate nebuliser suspensions in healthy Chinese subjects


> Added to model reports.

Total price for extracting content from 114 papers: $0.262


### Total costs

Now we can calculate the total costs for the flagging and the extraction of structured information from all suitable publications.

In [13]:
# define prices for these calls (in USD)
total_price = total_price_flag + total_price_content
print(f"Total price of this notebook execution: ${total_price:.3f}")

Total price of this notebook execution: $0.314
